In [1]:
############ LSTM on imdb

"""
Deeper LSTM on a built-in TensorFlow dataset (No custom functions/classes)
Task: Classify movie reviews as Positive or Negative
Dataset: IMDB (built into Keras, auto-downloads on first run)
Extra layers: second stacked LSTM, extra Dense layer, extra Dropout
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# STEP 1: Load the built-in dataset

vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

C:\Users\Mohana\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\datasets\npz_utils.py:68: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return RestrictedUnpickler(fp).load()


Training samples: 25000
Testing samples: 25000


In [3]:
# STEP 2: Pad sequences to the same length
max_length = 200
X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)


In [4]:
# STEP 3: Build the deeper LSTM model
model = models.Sequential([
    # Embedding: each word-integer becomes a learned 64-number vector
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),

    # Dropout on word vectors so the model doesn't memorize specific words
    layers.Dropout(0.3),

    # EXTRA LAYER 1: first LSTM with return_sequences=True, so it passes
    # its output at every time step to the next LSTM (not just the last)
    layers.LSTM(64, return_sequences=True),

    # EXTRA LAYER 2: second LSTM builds higher-level patterns on top of
    # what the first LSTM found, then outputs one summary vector
    layers.LSTM(32),

    layers.Dropout(0.4),

    # EXTRA LAYER 3: additional Dense layer to refine the features
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),

    # Output: one probability (0 = Negative, 1 = Positive)
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


C:\Users\Mohana\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\core\embedding.py:119: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# STEP 4: Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=4,
    batch_size=64
)

Epoch 1/4
391/391 ━━━━━━━━━━━━━━━━━━━━ 132s 319ms/step - accuracy: 0.7803 - loss: 0.4639 - val_accuracy: 0.8576 - val_loss: 0.3482
Epoch 2/4
391/391 ━━━━━━━━━━━━━━━━━━━━ 129s 303ms/step - accuracy: 0.8954 - loss: 0.2705 - val_accuracy: 0.8378 - val_loss: 0.3650
Epoch 3/4
391/391 ━━━━━━━━━━━━━━━━━━━━ 151s 327ms/step - accuracy: 0.9271 - loss: 0.2047 - val_accuracy: 0.8561 - val_loss: 0.3429
Epoch 4/4
391/391 ━━━━━━━━━━━━━━━━━━━━ 126s 322ms/step - accuracy: 0.9316 - loss: 0.1874 - val_accuracy: 0.8570 - val_loss: 0.3468


In [6]:
# STEP 5: Evaluate on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)


782/782 ━━━━━━━━━━━━━━━━━━━━ 58s 74ms/step - accuracy: 0.8570 - loss: 0.3468
Test Accuracy: 0.8570399880409241


In [7]:
# STEP 6: Predict on your own custom review
word_index = imdb.get_word_index()

review_text = "this movie was fantastic and the acting was brilliant"

encoded = [1]   # 1 = start-of-review token
for word in review_text.split():
    index = word_index.get(word, 2) # 2 = unknown word
    if index + 3 < vocab_size:
        encoded.append(index + 3)
    else:
        encoded.append(2)

new_review = pad_sequences([encoded], maxlen=max_length)

prediction = model.predict(new_review)[0][0]
label = "Positive" if prediction > 0.5 else "Negative"

print("Review:", review_text)
print("Prediction score:", prediction)
print("Predicted sentiment:", label)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 788ms/step
Review: this movie was fantastic and the acting was brilliant
Prediction score: 0.7783364
Predicted sentiment: Positive
